# Lo que escribes tú, traducido a alias (P1-b.3a)
Si escribes un valor real en una pregunta o en una nota, el agente lo traduce a su alias **antes** de enviarlo al modelo, y te
enseña lo enviado. `preview` lo hace sin llamar al modelo (no gasta API). Ver `docs/privacidad.md`.

In [ ]:
%load_ext autoreload
%autoreload 2
from dfir_copilot.agent.graph import build_agent
from dfir_copilot.agent.llm import make_llm
from dfir_copilot.cases import CaseWorkspace
from dfir_copilot.privacy import AmbiguousText

ws = CaseWorkspace.open("IDOR-INVOICES-2020Q4-TZ-SANTIAGO")
real = ws.engine(timeout_s=120)
engine, ps = ws.pseudonymized(timeout_s=120)
ledger = ws.ledger(real)
agent = build_agent(engine, ledger, make_llm(), max_steps=12)   # abre el diccionario de la copia por su cuenta

## 1. Vista previa: qué recibiría el modelo
Los valores reales se sacan del diccionario en tiempo de ejecución, así el notebook no los lleva escritos.

In [ ]:
ip_real = ps.reveal("src_ip", "IP-1806")
user_real = ps.reveal("user_id", "U-0032")
q = f"¿Qué otras cuentas usaron {ip_real} y qué hizo {user_real}?"
res = agent.preview(q)
print("Escribiste (solo local):", q)
print("Recibiría el modelo:    ", res.text)
print("Sustituciones:", res.substitutions)
print("Ida y vuelta correcta:", ps.reveal_any(res.text) == q)

## 2. Lo ambiguo se bloquea, con la pista
Un id de factura real (número largo) no se puede traducir a ciegas: podría ser otra cifra. El mensaje dice cómo escribirlo.

In [ ]:
factura = ps.reveal("x_invoice_id", 123)          # un id real; el modelo lo ve como 123
try:
    agent.preview(f"¿Quién consultó la factura {factura}?")
except AmbiguousText as exc:
    print(exc)
print("Escrito como lo ve el modelo:", agent.preview(f"¿Quién consultó la factura {ps.alias('x_invoice_id', factura)}?").text)
print("Con literal (si fuera una cifra corriente):", agent.preview(f"Hay {factura} filas", literal=(str(factura),)).text)

## 3. Preguntar de verdad (gasta API)
Descomenta para probarlo. `r.sent` es exactamente lo que recibió el modelo; la respuesta viene en alias y `reveal_any` la traduce
**solo para ti**. Recuerda que un alias suelto en una lista puede abreviarse con un rango: no lo des por real, enuméralos.

In [ ]:
# r = agent.ask(q)
# print("Enviado al modelo:", r.sent)
# print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens)
# print(ps.reveal_any(r.answer) if r.answer else "(pausado: hay hipótesis pendientes de tu aprobación)")

## 4. Lo que quedó en el ledger
`question` es lo enviado (en alias) y los recuentos no llevan valores.

In [ ]:
for e in ledger.entries("agent_turn")[-3:]:
    d = e["data"]
    print(e["seq"], d["status"], "| sustituciones:", d.get("text_substitutions"), "| literales:", d.get("text_literal"))
    print("   ", (d["question"] or "")[:160])